In [ ]:
import sys
sys.path.append('../Bayesflow/')
import bayesflow as bf
import numpy as np
import numba
import pandas as pd
from numba import njit
import keras
import matplotlib.pyplot as plt
import seaborn as sns
import notebook
import pandas as pd
import pickle

In [ ]:
from simulator_simple import meta, prior, ddm, likelihood

# Start with simulators

In [ ]:
prior_simulator = bf.simulators.LambdaSimulator(prior)
likelihood_simulator = bf.simulators.LambdaSimulator(likelihood)
simulator = bf.simulators.make_simulator([prior_simulator, likelihood_simulator], meta_fn = meta)

In [ ]:
prior_samples = simulator.sample(500)

In [ ]:
param_names = ["Drift rate", "Boundary", "Non-decision time"]
param_keys = ["v", "a", "t0"]

In [ ]:
f = bf.diagnostics.plots.pairs_samples(prior_samples,
    variable_keys = param_keys,
    variable_names = param_names
)

In [ ]:
@keras.saving.register_keras_serializable(package="custom_package")
def forward_transform(n):
    return np.sqrt(n)

@keras.saving.register_keras_serializable(package="custom_package")
def inverse_transfrom(n):
    return n ** 2

In [ ]:
adapter = (
    bf.Adapter()
    .broadcast("n_obs", to="rts")
    .as_set(["rts", "timeouts"])
    .standardize(exclude=["n_obs", "timeouts"])
    .apply(include="n_obs", forward=forward_transform, inverse=inverse_transfrom)
    .convert_dtype(from_dtype="float64", to_dtype="float32")
    .concatenate(["v", "a", "t0"], into="inference_variables")
    .concatenate(["rts", "timeouts"], into="summary_variables")
    # .rename("rts", "summary_variables")
    .rename("n_obs", "inference_conditions")
)

In [ ]:
summary_net = bf.networks.DeepSet(summary_dim=8, depth=2)
inference_net = bf.networks.CouplingFlow(depth=4)

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_net,
    summary_network=summary_net,
    initial_learning_rate=5e-4,
    optimizer=keras.optimizers.Adam,#AdamW,
    inference_variables=["v", "a", "t0"],
    summary_variables=["rts", "timeouts"],
    inference_conditions=["n_obs"],
    checkpoint_filepath=f"checkpoints/simple_ddm",
    checkpoint_name="one_bound_simple_ddm",    
)

In [ ]:
history = workflow.fit_online(epochs=150, 
                              batch_size=32)

In [ ]:
# for later reloading the checkpoints:
checkpoint_path=f"checkpoints/simple_ddm/one_bound_simple_ddm.keras"
workflow.approximator = keras.saving.load_model(checkpoint_path)

In [ ]:
test_sims = workflow.simulate(300)

In [ ]:
workflow.plot_diagnostics(test_sims)

In [ ]:
f = bf.diagnostics.plots.loss(history)
f.savefig("plots/loss.pdf")

In [ ]:
num_samples = 1000

test_sims = workflow.simulate(300)
data = test_sims.pop("rts")
n_obs = test_sims.pop("n_obs")
timeouts = test_sims.pop('timeouts')

samples = workflow.sample(conditions={"rts":data, "n_obs":n_obs, "timeouts":timeouts}, num_samples=num_samples)

In [ ]:
f = bf.diagnostics.plots.calibration_ecdf(samples, 
                                          test_sims,
                                          variable_names=param_names,
                                          difference=True)

f.savefig("plots/calibration_ecdf_simple.pdf")

In [ ]:
f = bf.diagnostics.plots.calibration_histogram(samples, test_sims, variable_names=param_names)
f.savefig("plots/calib_hist_simple.pdf")

In [ ]:
f = bf.diagnostics.plots.recovery(samples, test_sims,  variable_names=param_names)
f.savefig("plots/plot_recovery_simple.pdf")

# Prior RT simulations

In [ ]:
prior_samples = simulator.sample(10000)

In [ ]:
sns.kdeplot(prior_samples["rts"].mean(axis=1), fill=True)

In [ ]:
sns.kdeplot(prior_samples["rts"].std(axis=1), fill=True)

In [ ]:
from scipy.stats import skew

In [ ]:
sns.kdeplot(skew(prior_samples["rts"], axis=1), fill=True)

### 1. Posterior retrodictive Check

In [ ]:
num_samples = 1000
num_sim = 10
num_resim = 50

sims = workflow.simulate(num_sim)
rts = sims.pop("rts")
n_obs = sims.pop("n_obs")
timeouts = sims.pop("timeouts")

# fit model and draw 1000 samples per dataset
post_samples = workflow.sample(conditions={"rts":rts, "n_obs":n_obs, "timeouts":timeouts}, num_samples=num_samples)

In [ ]:
index_set = np.random.choice(np.arange(num_samples), size=num_resim)

In [ ]:
# get mean values for each participant
post_samples_mean = {k: np.mean(t, axis=1) for k, t in post_samples.items()}

In [ ]:
v = post_samples.pop('v')
a = post_samples.pop('a')
t0 = post_samples.pop('t0')
post_samples_np = np.concatenate([v, a, t0], axis=2)

In [ ]:
# Re-simulate
pred_data = np.zeros((num_sim, num_resim, n_obs, 1))

for sim in range(num_sim):
    # for i, idx in enumerate(index_set):
    for i, idx in enumerate(index_set):
        v, a, t0 = post_samples_np[sim, idx, :]
        rt = ddm(v, a, t0, n_obs)
        pred_data[sim, i, :, :] = rt[:, np.newaxis]
        # pred_data[sim, i, :, :] = ddm(post_samples_np[sim, idx], n_obs)

In [ ]:
# Check simulated RTs distributions
f, axarr = plt.subplots(2, 5, figsize=(12, 4))
for i, ax in enumerate(axarr.flat):
    sns.histplot(rts[i, :].flatten(), ax=ax)
    sns.despine(ax=ax)
    ax.set_label("")
    ax.set_yticks([])
    if i > 4:
        ax.set_xlabel("Sim RTs")
f.tight_layout()

In [ ]:
f, axarr = plt.subplots(2, 4, figsize=(18, 8))
for i, ax in enumerate(axarr.flat):
    sns.kdeplot(
        rts[i, :].flatten(), ax=ax, fill=True, color="black", alpha=0.5, label="sim_data")
    sns.kdeplot(
        pred_data[i, :, :, 0].flatten(), ax=ax, fill=True, color="maroon", alpha=0.5, label="pred_data")
    ax.set_xlim((0, 5))
    ax.set_ylabel("")
    ax.set_yticks([])
    ax.set_title(f"Simulated data set #{i+1}", fontsize=18)

    # Set legeng to first plot
    if i==0:
        ax.legend()
    if i > (num_sim//2)-1:
        ax.set_xlabel("Response time (secs)", fontsize=16)
    sns.despine()
f.tight_layout()

### 2. Posterior Predictive check

# READ THE DATA

In [ ]:
# DON'T RUN THIS AGAIN!!
# Prepare data from HC3
data = pd.read_csv("datasets/HC3_simple.csv")
subs = data.RELEASEID.unique()
# N_SUBS = subs[0:3]
# N_SUBS = len(subs)
#N_SUBS = 10

subject_dicts = {}

for sub in subs:
    person_data = data[data.RELEASEID == sub]
    rt = np.array(person_data.TIME)[np.newaxis, :]
    n_obs = rt.shape[1]
    timeouts = np.zeros(n_obs) # added with no timeouts in subject data
    subject_dicts[sub] = {"rts":rt,
                          "n_obs":n_obs,
                          "timeouts":timeouts}

with open("datasets/HC3_simple.pkl", "wb") as f:
    pickle.dump(subject_dicts, f)

## fitting the model

In [ ]:
# the subject dictionary is saved, load it here:
with open("datasets/HC3_simple.pkl", "rb") as t:
    subject_dicts = pickle.load(t)

In [ ]:
# get the posterior samples
num_samples = 1000
subs=list(subject_dicts.keys())
# subs_small = subs[0:10]

posteriors = {}
for sub in subs:
    n_obs = subject_dicts[sub]["n_obs"]
    data  = subject_dicts[sub]["rts"]
    timeouts = subject_dicts[sub]["timeouts"]
    post = workflow.sample(conditions={"rts":data, "n_obs":n_obs, "timeouts":timeouts}, num_samples=num_samples)
    posteriors[sub] = post

In [ ]:
with open("datasets/postHC3_simple_all.pkl", "wb") as f:
    pickle.dump(posteriors, f)

In [ ]:
# load posteriors again
with open("datasets/postHC3_simple_all.pkl", "rb") as f:
    posteriors = pickle.load(f)

In [ ]:
# GET POSTERIOR SAMPLES' SUMMARIES
posterior_means = {}

for sub in posteriors.keys():
    posterior_means[sub] = {param: {"mean": np.mean(posteriors[sub][param]),
                                    "std" : np.std(posteriors[sub][param], ddof=1)}
                                    for param in posteriors[sub].keys()}

In [ ]:
with open("datasets/simple_postHC3_sum.pkl", "wb") as f:
    pickle.dump(posterior_means, f)

In [ ]:
with open("datasets/simple_postHC3_sum.pkl", "rb") as f:
    posterior_means = pickle.load(f)

In [ ]:
# save dictionary as dataframe 
sum_dataHC3simple = []
for sub, params in posterior_means.items():
    for param, stats in params.items():
        sum_dataHC3simple.append({
            "RELEASEID": sub,
            "param":param,
            "mean":stats["mean"],
            "std":stats["std"]
        })
df = pd.DataFrame(sum_dataHC3simple)

In [ ]:
# save it as csv for further analysis
df.to_csv("datasets/postsumHC3_simple.csv", index=False)